# SoccerMaster — example notebook

This walks through the refactored `soccermaster` package: install it, fetch the
pretrained weights, load the model, run inference, then run the full game-state
pipeline on a video and get the extracted data out.

It is written to run on a GPU box — Colab, Kaggle or a rented pod. Anything with a
CUDA device and a Python kernel works.

## What is verified

Run end to end on a Colab T4 (Python 3.13.15, torch 2.11.0+cu130,
transformers 4.51.3):

- `pip install` from GitHub, including the `pipeline` extra
- `import soccermaster` and its paths
- `load_config()` and the weight manifest
- downloading the SigLIP2 backbone + SoccerMaster checkpoints
- `load_model()` — 967.8 M parameters, all five heads, onto `cuda:0`
- `predict(..., task="detection")` — returns `pred_logits`, `pred_boxes`,
  `pred_roles`, `pred_jn_holistic`, `pred_digit_head`, `pred_digit_tail`, `outputs`

Verified locally with synthetic data:

- hydra config discovery — both `gsr_step_1_example` and
  `gsr_step_3_example_accelerate` compose, and every `_target_` resolves
- `export_state` against a state built to the real schema — CSV flattening, JSONL
  nesting, per-tracklet consensus

**Not verified:** the three game-state steps have never been executed end to end.
They need the `yolo` and `qwen7b` weights. Expect to fix small things on the first
real run — `soccermaster doctor` (section 8) is there to make that list short.

Cells are **guarded** so nothing multi-gigabyte downloads or runs by accident.

## If something breaks

Most first-run failures so far have been dependency versions rather than model logic —
the troubleshooting section at the end lists every one hit while getting this far.


In [2]:
# Install the package.
#
# `pipeline` is the extra that carries everything the game-state steps import at
# module level (torchreid, prtreid, SoccerNet, poseval, pycocotools, ...). Without it
# you can still import the package and load the model, but the pipeline will raise
# ModuleNotFoundError. Use "" for the lean install.
#
# This cell installs from the LOCAL checkout when there is one, so unpushed fixes are
# the thing being tested. Otherwise it installs from GitHub, which is what you want on
# Colab/Kaggle.
#
# CAUTION on hosted runtimes: they ship a CUDA-matched torch, and a dependency with an
# upper torch bound would downgrade it and break CUDA. That is why this project pins
# floors only (torch>=2.4). Nothing here force-reinstalls without --no-deps, so the
# runtime's own torch is never touched.

import importlib.util
import subprocess
import sys
from pathlib import Path

GIT_URL = "https://github.com/SurrealisticRabbit/SoccerMaster.git"
GIT_REF = "main"
EXTRAS = "pipeline"  # "" for the lean install, "pipeline,sam2" for step 2 too

# Is this notebook running inside a checkout of the project?
ROOT = None
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate = parent / "pyproject.toml"
    if candidate.is_file() and "soccermaster" in candidate.read_text(errors="ignore"):
        ROOT = parent
        break

if ROOT is not None:
    SPEC = f"{ROOT}[{EXTRAS}]" if EXTRAS else str(ROOT)
else:
    SPEC = f"soccermaster[{EXTRAS}] @ git+{GIT_URL}@{GIT_REF}"

ALREADY = importlib.util.find_spec("soccermaster") is not None

print(f"python        : {sys.executable} ({sys.version.split()[0]})")
print(f"local checkout: {ROOT}")
print(f"already there : {ALREADY}")
print(f"installing    : {SPEC}\n")


def pip(*args):
    """Run pip and show what it did; raise with the real error on failure."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-1500:])
    if result.returncode:
        print(result.stderr[-3000:])
        raise RuntimeError(f"pip failed (exit {result.returncode}): {' '.join(cmd)}")


if ROOT is not None:
    # editable install of the checkout; dependencies resolve normally
    pip("-e", SPEC)
elif ALREADY:
    # pip will not replace an already-installed direct URL whose version string is
    # unchanged, so a plain re-run would silently keep the old code. Force the code in
    # first, without touching dependencies (and so without touching torch)...
    pip("--force-reinstall", "--no-deps", "--upgrade", SPEC)
    # ...then install normally so any newly-declared dependency is picked up.
    pip("--upgrade", SPEC)
else:
    pip("--upgrade", SPEC)

print("done")


python        : /usr/bin/python3 (3.13.15)
local checkout: None
already there : True
installing    : soccermaster[pipeline] @ git+https://github.com/SurrealisticRabbit/SoccerMaster.git@main

SoccerMaster.git@main (from soccermaster[pipeline]@ git+https://github.com/SurrealisticRabbit/SoccerMaster.git@main)
  Cloning https://github.com/SurrealisticRabbit/SoccerMaster.git (to revision main) to /tmp/pip-install-6tj9r5au/soccermaster_d16ce8686bfb45bcbf155ccdec185299
  Resolved https://github.com/SurrealisticRabbit/SoccerMaster.git to commit 5873fd94911fbf8f13e7043c9b4f2577ff651b84
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for soccermaster: filename=soccermaster-0.1.0-py3-none-any.whl size=10

## 1. Environment and where things get stored

Everything the package reads and writes lives under a single root —
`SOCCERMASTER_HOME`. You must set it **before** the paths are first used so that
weights and outputs land somewhere you can find (and, on Colab, somewhere you know
is ephemeral unless you mount Drive).

Run this cell, then **restart the runtime** if you set a new value, so every
subsequent import sees it.


In [3]:
import os
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Where should weights/, data/ and outputs/ live?
# ---------------------------------------------------------------------------
USE_DRIVE = False  # set True on Colab to persist weights across sessions

try:
    import google.colab  # noqa: F401

    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if USE_DRIVE and ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    SM_HOME = Path("/content/drive/MyDrive/soccermaster")
elif ON_COLAB:
    # /content is wiped when the runtime recycles -- fine for a one-off run.
    SM_HOME = Path("/content/soccermaster")
else:
    SM_HOME = Path.cwd() / "soccermaster_home"

SM_HOME.mkdir(parents=True, exist_ok=True)
os.environ["SOCCERMASTER_HOME"] = str(SM_HOME)

print(f"python      : {sys.version.split()[0]}")
print(f"on colab    : {ON_COLAB}")
print(f"SM_HOME     : {SM_HOME}")

try:
    import torch

    print(f"torch       : {torch.__version__}")
    print(f"cuda avail  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"gpu         : {torch.cuda.get_device_name(0)}")
except ImportError:
    print("torch       : NOT INSTALLED -- install a CUDA build before continuing")


python      : 3.13.15
on colab    : True
SM_HOME     : /content/soccermaster
torch       : 2.11.0+cu130
cuda avail  : True
gpu         : Tesla T4


## 2. Verify the install

`import soccermaster` is deliberately dependency-free: the public API is re-exported
lazily, so this import succeeds even if nothing else is installed. That makes it a
quick check that the package itself landed.


In [4]:
import soccermaster as sm

print("soccermaster :", sm.__version__)
print("home         :", sm.paths.home())
print("weights      :", sm.paths.weights_dir())
print("data         :", sm.paths.data_dir())
print("outputs      :", sm.paths.outputs_dir())

# The public surface: everything a notebook should need.
print("api          :", ", ".join(sm.__all__))


soccermaster : 0.1.0
home         : /content/soccermaster
weights      : /content/soccermaster/weights
data         : /content/soccermaster/data
outputs      : /content/soccermaster/outputs
api          : analyze_video, load_config, load_model, predict, prepare, prepare_all, run_pipeline, paths


## 3. What weights are needed

`soccermaster prepare` reads a manifest and puts each set where the configs expect
it. Two entries cannot be fetched automatically and say so.


In [5]:
from soccermaster.downloads import load_manifest

manifest = load_manifest()
print(f"{'name':<14}{'kind':<9}{'source':<9}destination")
print("-" * 78)
for name, entry in manifest.items():
    # kind: auto = `soccermaster prepare` fetches it
    #       runtime = the pipeline fetches it itself on first use
    #       MANUAL = a human must place the file
    print(f"{name:<14}{entry.kind:<9}{entry.source:<9}{entry.dest}")

# Which of them are already on disk?
print("\nAlready present:")
for name, entry in manifest.items():
    mark = "yes" if entry.dest.exists() and any(entry.dest.iterdir()) else "no "
    print(f"  {name:<14}{mark}")


name          kind    source   destination
------------------------------------------------------------------------------
siglip        auto    hf       /content/soccermaster/weights/google/siglip2-large-patch16-512
soccermaster  auto    hf       /content/soccermaster/weights/SoccerMaster
yolo          auto    hf       /content/soccermaster/weights/yolo
qwen7b        auto    hf       /content/soccermaster/weights/jn/Qwen2.5-VL-7B-Instruct
qwen72b       auto    hf       /content/soccermaster/weights/jn/Qwen2.5-VL-72B-Instruct
sam2          auto    hf       /content/soccermaster/weights/sam2
legibility    auto    gdrive   /content/soccermaster/weights/legibility
reid          MANUAL  runtime  /content/soccermaster/weights/reid
calibration   MANUAL  runtime  /content/soccermaster/weights/calibration

Already present:
  siglip        no 
  soccermaster  no 
  yolo          no 
  qwen7b        no 
  qwen72b       no 
  sam2          no 
  legibility    no 
  reid          no 
  calibration 

## 4. Download the weights

Multi-gigabyte. The cell is guarded by `DOWNLOAD`; the `dry_run` pass below prints
exactly what would be fetched without touching the network.

For **inference only** you need `siglip` (backbone) + `soccermaster` (checkpoints).
Add `yolo` for step 1 of the pipeline, `qwen7b` for step 3, `sam2` for step 2.


In [ ]:
DOWNLOAD = False  # flip to True to actually fetch the weights

# What would happen, without downloading anything:
for name in ("siglip", "soccermaster", "yolo"):
    sm.prepare(name, dry_run=True)

if DOWNLOAD:
    sm.prepare("siglip")        # SigLIP2 backbone (Google) -- needed to build the model
    sm.prepare("soccermaster")  # SoccerMaster checkpoints  -- needed for inference
    sm.prepare("yolo")          # person detector          -- needed for pipeline step 1
    sm.prepare("qwen7b")        # jersey/role (Alibaba)    -- needed for pipeline step 3
    # sm.prepare("sam2")        # SAM 2 checkpoint         -- needed for pipeline step 2
    #
    # `reid` and `calibration` are NOT fetched here. The pipeline downloads those
    # itself from Zenodo on first run (md5-verified), so there is no manual step.
else:
    print("DOWNLOAD is False - nothing fetched.")

# For step 1 specifically, the only weight you must have is the YOLO detector: its
# wrapper does `YOLO(cfg.path_to_checkpoint)` and does not download. The manifest
# stores it under the name the configs expect:
#   weights/yolo/yolo_v8x6_person_lr_default_best.pt
print("\nstep-1 gating weight:",
      sm.paths.weights_dir() / "yolo" / "yolo_v8x6_person_lr_default_best.pt")


[siglip] source=hf dest=/content/soccermaster/weights/google/siglip2-large-patch16-512
  SigLIP2 backbone used by the SoccerMaster encoder.
  would download hf://google/siglip2-large-patch16-512 (allow=*)
[soccermaster] source=hf dest=/content/soccermaster/weights/SoccerMaster
  SoccerMaster pretrained checkpoints (backbone + task heads).
  would download hf://xleprime/SoccerMaster (allow=*)
[yolo] source=hf dest=/content/soccermaster/weights/yolo
  Person detector used by the GSR step-1 pipeline.
  would download hf://xleprime/SoccerMaster (allow=['yolo_v8x6_finetuned.pt', 'yolo/*'])
DOWNLOAD is False - nothing fetched.


## 5. Inspect the config (no weights needed)

Cheap sanity check. This also shows how paths are resolved: `CKPT_PATH` is rewritten
to point under `weights_dir()`, so the configs no longer depend on the working
directory.


In [ ]:
cfg = sm.load_config("pretrain")

print(f"config keys : {len(cfg)}")
for key in (
    "SIGLIP_BACKBONE_TYPE",
    "NUM_FRAMES",
    "AUG_MAX_SIZE",
    "TEMPORAL_START_LAYER",
    "CKPT_PATH",
    "TEXT_ENCODER_CKPT_PATH",
):
    print(f"{key:<22}: {cfg.get(key)}")

print("\nDATASETS_TO_HEADS (task groups -> heads):")
for dataset, heads in cfg["DATASETS_TO_HEADS"].items():
    print(f"  {dataset:<24} {heads}")


config keys : 182
SIGLIP_BACKBONE_TYPE  : soccer_master
NUM_FRAMES            : 30
AUG_MAX_SIZE          : 512
TEMPORAL_START_LAYER  : 16
CKPT_PATH             : /content/soccermaster/weights/google/siglip2-large-patch16-512
TEXT_ENCODER_CKPT_PATH: /content/soccermaster/weights/google/siglip2-large-patch16-512

DATASETS_TO_HEADS (task groups -> heads):
  SoccerNetGSR_Detection   ['SoccerNetGSR_Detection', 'LinesDetection', 'KeypointsDetection']
  VideoCaption             ['VideoCaption', 'CaptionClassification']


## 6. Load the model

`load_model()` builds `MultiTaskingModel` and loads the backbone plus the task heads
from `weights/SoccerMaster`. Guarded so the cell explains itself instead of raising a
confusing error when the weights are absent.


In [ ]:
model = None

checkpoint_dir = sm.paths.weights_dir() / "SoccerMaster"
backbone_dir = cfg["CKPT_PATH"]

missing = [p for p in (checkpoint_dir, backbone_dir) if not Path(p).exists()]

if missing:
    print("Weights not found yet:")
    for p in missing:
        print(f"  {p}")
    print("\nSet DOWNLOAD = True in the download cell and re-run it, or place the")
    print("files manually. The rest of the notebook needs these.")
else:
    model = sm.load_model()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"model      : {type(model).__name__}")
    print(f"parameters : {n_params/1e6:.1f} M")
    print(f"device     : {next(model.parameters()).device}")
    print(f"heads      : {list(model.multi_task_head.keys())}")


Using Backbone type: soccer_master
Loaded backbone weights from: /content/soccermaster/weights/SoccerMaster/backbone.pt
Loading CaptionClassification head from: /content/soccermaster/weights/SoccerMaster/CaptionClassification.pt
Loading KeypointsDetection head from: /content/soccermaster/weights/SoccerMaster/KeypointsDetection.pt
Loading LinesDetection head from: /content/soccermaster/weights/SoccerMaster/LinesDetection.pt
Loading SoccerNetGSR_Detection head from: /content/soccermaster/weights/SoccerMaster/SoccerNetGSR_Detection.pt
Loading VideoCaption head from: /content/soccermaster/weights/SoccerMaster/VideoCaption.pt
model      : MultiTaskingModel
parameters : 967.8 M
device     : cuda:0
heads      : ['CaptionClassification', 'KeypointsDetection', 'LinesDetection', 'SoccerNetGSR_Detection', 'VideoCaption']


## 7. Run inference

`predict(model, frames, task=...)` takes either:

- a directory — `.../SNGS-10001` or `.../SNGS-10001/img1` (a flat folder of images
  also works), or
- an array of `uint8` HWC frames, or a list of such arrays.

It returns the **raw head outputs** (no NMS, no bbox rescaling), matching the
upstream `inference_demo.py`. `task` is one of `detection`, `lines`, `keypoints`,
`caption`, `classification` or `all`.

The cell below uses random frames so you can exercise the forward pass without any
dataset. Point `FRAMES` at a real sequence to get meaningful output.


In [ ]:
import numpy as np

# Point this at a real sequence when you have one, e.g.
#   sm.paths.data_dir() / "SoccerNetGS" / "sn500" / "SNGS-10001"
FRAMES = None

if model is None:
    print("Load the model first (previous cell).")
else:
    if FRAMES is None:
        # The temporal encoder expects cfg["NUM_FRAMES"] frames.
        n = cfg["NUM_FRAMES"]
        FRAMES = np.random.randint(0, 256, size=(n, 540, 960, 3), dtype=np.uint8)
        print(f"using {n} random frames at {FRAMES.shape[1]}x{FRAMES.shape[2]}")

    try:
        outputs = sm.predict(model, FRAMES, task="detection")
    except Exception as exc:  # surface the real error, this path is unverified
        print(f"predict failed: {type(exc).__name__}: {exc}")
    else:
        def describe(value, indent="    "):
            import torch

            if isinstance(value, torch.Tensor):
                print(f"{indent}shape={tuple(value.shape)} dtype={value.dtype}")
            elif isinstance(value, dict):
                for k, v in value.items():
                    print(f"{indent}{k}:")
                    describe(v, indent + "  ")
            elif isinstance(value, (list, tuple)):
                print(f"{indent}{type(value).__name__} of length {len(value)}")
                for i, item in enumerate(value[:2]):
                    print(f"{indent}  [{i}]")
                    describe(item, indent + "    ")
            else:
                print(f"{indent}{value}")

        for head, head_output in outputs.items():
            print(f"[{head}]")
            describe(head_output)


using 30 random frames at 540x960
[SoccerNetGSR_Detection]
    pred_logits:
      shape=(1, 30, 300, 1) dtype=torch.float32
    pred_boxes:
      shape=(1, 30, 300, 4) dtype=torch.float32
    pred_roles:
      shape=(1, 30, 300, 6) dtype=torch.float32
    pred_jn_holistic:
      shape=(1, 30, 300, 101) dtype=torch.float32
    pred_digit_head:
      shape=(1, 30, 300, 10) dtype=torch.float32
    pred_digit_tail:
      shape=(1, 30, 300, 11) dtype=torch.float32
    aux_outputs:
      list of length 0
    outputs:
      shape=(1, 30, 300, 256) dtype=torch.float32


## 8. Pre-flight check (run this before the pipeline)

A dry run that answers "will the pipeline start, or am I about to hit a wall?" It
does not track anything, download anything, or need a GPU.

It checks two things:

1. **Which weight files are present** under `weights/`. `yolo` and `qwen7b` must be
   there; `reid` and `calibration` are fetched by the pipeline itself on first run,
   so their absence here is expected and fine.
2. **That the pre-flight's own checks pass** — in particular that every module named
   in the composed Hydra configs is importable. Each pipeline module is a dotted
   class path in YAML; a wrong one dies at instantiation with a confusing message.

Needs the `pipeline` extra for `hydra`.


## 9. The full game-state pipeline (frames already on disk)

Three steps:

| step | what it does | needs |
|---|---|---|
| 1 | person detection + tracking | `yolo` weights |
| 2 | SAM 2 mask refinement | `sam2` extra + `sam2` weights + CUDA |
| 3 | calibration, jersey number, role, team | `qwen7b` weights, `reid` + `calibration` weights |

This is the same Hydra configuration the upstream scripts used, driven for you: the
dataset path, the weight directory and the state files between steps are injected, so
it no longer depends on the working directory or on hardcoded sequence ids.

Use this when you already have frames in SoccerNet layout. If you have a **video**,
skip to the next section — `analyze_video` does the decoding and the export for you.

Steps 1 and 3 run on CPU or GPU. **Step 2 requires the `sam2` extra and a CUDA
build.**


In [ ]:
RUN_PIPELINE = False  # flip to True once the dataset and weights are in place

DATASET = sm.paths.data_dir() / "SoccerNetGS"   # <split>/<sequence>/img1 layout
OUT_DIR = sm.paths.outputs_dir() / "pipeline"
VIDEO_IDS = [10001, 10002]                      # sequences to process
STEPS = (1, 3)                                  # skip 2 unless the sam2 extra is installed

if RUN_PIPELINE:
    if not DATASET.is_dir():
        print(f"dataset not found: {DATASET}")
        print("Run: python -m soccermaster.model.data.extract_frames --help")
    else:
        # device is a CUDA_VISIBLE_DEVICES value, e.g. "0" or "0,1"
        out = sm.run_pipeline(
            DATASET,
            out_dir=OUT_DIR,
            steps=STEPS,
            video_ids=VIDEO_IDS,
            device="0",
        )
        print("outputs written to:", out)
        for path in sorted(out.rglob("*.pklz")):
            print(" ", path.relative_to(out), f"({path.stat().st_size/1e6:.1f} MB)")
else:
    print("RUN_PIPELINE is False - nothing run.")
    print(f"would read  : {DATASET}")
    print(f"would write : {OUT_DIR}")


RUN_PIPELINE is False - nothing run.
would read  : /content/soccermaster/data/SoccerNetGS
would write : /content/soccermaster/outputs/pipeline


**`reid` / `calibration` weights**
Not a manual step. The ReID module downloads
`prtreid-soccernet-baseline.pth.tar` and `hrnetv2_w32_imagenet_pretrained.pth` from
Zenodo on first run, and the calibration module downloads `SV_kp` / `SV_lines`. Both
are md5-verified and skip the download when the file is already present.


In [ ]:
# Pre-flight. Read-only: nothing is downloaded, tracked or loaded.
#
# `doctor` returns a non-zero exit code when something blocking is found, so this
# cell prints rather than raising -- a failure list is the useful output here.
import subprocess
import sys

DOCTOR_STEPS = "1,3"     # what you intend to run
DOCTOR_ARGS = ["--steps", DOCTOR_STEPS]

# Point this at your frames dir once you have one, to have the layout checked too:
DATASET = None
if DATASET:
    DOCTOR_ARGS += ["--dataset", str(DATASET)]

proc = subprocess.run(
    [sys.executable, "-m", "soccermaster", "doctor", *DOCTOR_ARGS],
    capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode:
    print(proc.stderr[-2000:])
    print("^ blocking problems above; fix those before running the pipeline")


## 10. Run your own video: data out


In [ ]:
# Upload a video (Colab/Kaggle) or point VIDEO at a local path, then run.
#
# On Colab this opens a file picker; on a RunPod/Jupyter box just set VIDEO directly.
import os
from pathlib import Path

VIDEO = None            # e.g. "/workspace/my_match.mp4"
VIDEO_ID = 10001        # becomes the sequence dir SNGS-10001
STEPS = (1, 3)          # 1 = detect+track, 3 = calibration/jersey/role/team
WRITE_VIDEO = True      # also render the annotated mp4

if VIDEO is None:
    try:
        from google.colab import files

        uploaded = files.upload()
        VIDEO = next(iter(uploaded))
        print("uploaded:", VIDEO)
    except Exception:
        print("set VIDEO to a path, or upload via the Colab file picker")

if VIDEO and Path(VIDEO).is_file():
    result = sm.analyze_video(
        VIDEO,
        video_id=VIDEO_ID,
        steps=STEPS,
        write_video=WRITE_VIDEO,
    )
    print("\n" + "=" * 60)
    print("DATA")
    for kind, path in result["exports"].items():
        print(f"  {kind:<18} {path}")
    if result["videos"]:
        print("VIDEO")
        for path in result["videos"]:
            print(f"  {path}")
else:
    print("nothing to analyse yet -- set VIDEO")


In [7]:
# TEMP test (removed after this run): does the game-state pipeline actually run?
#
# Builds a tiny synthetic dataset in the layout the dataset wrapper expects and runs
# step 1 (detection + tracking) on it. This exercises: config discovery, dataset
# loading, sequence filtering, the YOLO detector, ReID (which downloads its weights
# from Zenodo on first use), the tracker, state saving and the data export.
import subprocess
import sys
from pathlib import Path

import numpy as np
from PIL import Image

ROOT = sm.paths.data_dir() / "SoccerNetGS"
EVAL_SET = "sn500"
VIDEO_ID = 10001
N_FRAMES = 8

seq = ROOT / EVAL_SET / f"SNGS-{VIDEO_ID}"
img1 = seq / "img1"
img1.mkdir(parents=True, exist_ok=True)

# Frames have to look vaguely like a pitch or YOLO finds nothing and the tracker has
# nothing to track, which would "pass" while testing nothing. Draw a green pitch with
# a few person-like rectangles moving between frames.
for f in range(1, N_FRAMES + 1):
    out = img1 / f"{f:06d}.jpg"
    if out.exists():
        continue
    canvas = np.zeros((1080, 1920, 3), dtype=np.uint8)
    canvas[:, :] = (40, 110, 40)                     # grass
    for i in range(6):
        x = int(150 + i * 260 + f * 6)               # drift right each frame
        y = int(700 - i * 25)
        canvas[y:y + 150, x:x + 55] = (220, 220, 220)  # light shirt
        canvas[y + 60:y + 150, x:x + 55] = (40, 40, 90)  # dark shorts
    Image.fromarray(canvas).save(out, quality=95)

print(f"frames written: {len(list(img1.glob('*.jpg')))} -> {img1}")

# The dataset wrapper matches filter names against os.listdir(), so it is SNGS-<id>.
result = subprocess.run(
    [
        sys.executable, "-m", "soccermaster", "doctor",
        "--steps", "1", "--dataset", str(ROOT),
    ],
    capture_output=True, text=True,
)
print("\n===== DOCTOR =====")
print(result.stdout[-3000:])


frames written: 8 -> /content/soccermaster/data/SoccerNetGS/sn500/SNGS-10001/img1

===== DOCTOR =====
soccermaster pre-flight
steps checked: 1

Environment
-----------
  [ ok ] soccermaster 0.1.0 at /usr/local/lib/python3.13/dist-packages/soccermaster
  [note] home     /content/soccermaster
  [note] weights  /content/soccermaster/weights
  [note] data     /content/soccermaster/data
  [note] outputs  /content/soccermaster/outputs
  [ ok ] torch 2.11.0+cu130
  [ ok ] cuda available: Tesla T4

Third-party imports
-------------------
  [ ok ] hydra
  [ ok ] omegaconf
  [ ok ] torch
  [ ok ] torchreid
  [ ok ] ultralytics

Weights
-------
  [ ok ] yolo         present at /content/soccermaster/weights/yolo
  [note] reid         not downloaded yet; the pipeline fetches it from Zenodo on first run (md5-verified)
  [note] step-1 detector expects /content/soccermaster/weights/yolo/yolo_v8x6_person_lr_default_best.pt

Hydra configs
-------------
  [ ok ] gsr_step_1_example: composed, 19 keys, 3 p